**Purpose**: prepare PO-item-level pricing features for DB_05

**Modeling choice**: DB_04 will compute every historical benchmark using only transactions before the current PO item. The current price never contributes to its own benchmark.

**Imports and configuration**

In [0]:
# ============================================================
# DB_04_Pricing_Anomaly_Feature_Engineering
#
# Purpose:
# - Read Gold purchase-order data from Microsoft Fabric
# - Build leakage-safe historical pricing benchmarks
# - Compare PO price against:
#       historical material price
#       historical supplier-material price
#       historical category price
#       negotiated contract price
# - Create training and 2026 scoring feature datasets
# - Persist ML feature products to OneLake
#
# DB_05 will train the Isolation Forest model.
# ============================================================

from datetime import date, datetime, timezone

import math

from pyspark.sql import functions as F

from pyspark.sql import Window


# ------------------------------------------------------------
# Project dates
# ------------------------------------------------------------

AS_OF_DATE = date(
    2026,
    7,
    31
)

TRAINING_END_YEAR = 2025

SCORING_YEAR = 2026


# ------------------------------------------------------------
# Historical benchmark requirements
#
# Material prices are the strongest comparable benchmark.
# Supplier-material history is more specific but naturally
# sparser.
# Category history is broad, so we require more observations.
# ------------------------------------------------------------

MIN_MATERIAL_HISTORY_COUNT = 5

MIN_SUPPLIER_MATERIAL_HISTORY_COUNT = 3

MIN_CATEGORY_HISTORY_COUNT = 20


# ------------------------------------------------------------
# Diagnostic rule thresholds
#
# These do NOT train Isolation Forest.
# They are only a transparent business-rule proxy that can
# later be used to sanity-check anomaly-model behavior.
# ------------------------------------------------------------

EXTREME_PRICE_VARIANCE_PCT = 20.0

EXTREME_Z_SCORE = 3.0


print(
    "DB_04 configuration loaded."
)

print(
    "As-of date:",
    AS_OF_DATE
)

print(
    "Historical training through:",
    TRAINING_END_YEAR
)

print(
    "Scoring year:",
    SCORING_YEAR
)

print(
    "Minimum material history:",
    MIN_MATERIAL_HISTORY_COUNT
)

print(
    "Minimum supplier-material history:",
    MIN_SUPPLIER_MATERIAL_HISTORY_COUNT
)

print(
    "Minimum category history:",
    MIN_CATEGORY_HISTORY_COUNT
)

DB_04 configuration loaded.
As-of date: 2026-07-31
Historical training through: 2025
Scoring year: 2026
Minimum material history: 5
Minimum supplier-material history: 3
Minimum category history: 20


**Load OneLake credentials**

In [0]:
# ============================================================
# Load Fabric OneLake credentials securely
# ============================================================

tenant_id = dbutils.secrets.get(
    scope="fabric-onelake",
    key="fabric-tenant-id"
)

client_id = dbutils.secrets.get(
    scope="fabric-onelake",
    key="fabric-client-id"
)

client_secret = dbutils.secrets.get(
    scope="fabric-onelake",
    key="fabric-client-secret"
)


print(
    "Fabric OneLake credentials loaded securely."
)

Fabric OneLake credentials loaded securely.


**Configure OneLake OAuth**

In [0]:
# ============================================================
# Configure Fabric OneLake OAuth
# ============================================================

spark.conf.set(
    "fs.azure.account.auth.type",
    "OAuth"
)

spark.conf.set(
    "fs.azure.account.oauth.provider.type",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    "fs.azure.account.oauth2.client.id",
    client_id
)

spark.conf.set(
    "fs.azure.account.oauth2.client.secret",
    client_secret
)

spark.conf.set(
    "fs.azure.account.oauth2.client.endpoint",
    (
        f"https://login.microsoftonline.com/"
        f"{tenant_id}/oauth2/token"
    )
)


print(
    "OneLake OAuth configuration applied."
)

OneLake OAuth configuration applied.


**Define Gold and ML paths**

In [0]:
# ============================================================
# Fabric Gold and Pricing ML paths
# ============================================================

FACT_SUPPLIER_PERFORMANCE_PATH = (
    "abfss://Project_Procurement@onelake.dfs.fabric.microsoft.com/lh_procurement_gold.Lakehouse/Tables/fact_supplier_performance"
)


# ------------------------------------------------------------
# Derive Gold Lakehouse root
# ------------------------------------------------------------

GOLD_LAKEHOUSE_ROOT = (
    FACT_SUPPLIER_PERFORMANCE_PATH
    .rsplit(
        "/Tables/",
        1
    )[0]
)


# ------------------------------------------------------------
# Gold analytical inputs
# ------------------------------------------------------------

FACT_PURCHASE_ORDER_PATH = (
    f"{GOLD_LAKEHOUSE_ROOT}/"
    f"Tables/fact_purchase_order"
)

DIM_SUPPLIER_PATH = (
    f"{GOLD_LAKEHOUSE_ROOT}/"
    f"Tables/dim_supplier"
)

DIM_MATERIAL_PATH = (
    f"{GOLD_LAKEHOUSE_ROOT}/"
    f"Tables/dim_material"
)

DIM_CATEGORY_PATH = (
    f"{GOLD_LAKEHOUSE_ROOT}/"
    f"Tables/dim_category"
)

DIM_CONTRACT_PATH = (
    f"{GOLD_LAKEHOUSE_ROOT}/"
    f"Tables/dim_contract"
)

DIM_DATE_PATH = (
    f"{GOLD_LAKEHOUSE_ROOT}/"
    f"Tables/dim_date"
)


# ------------------------------------------------------------
# ML working area
# ------------------------------------------------------------

PRICING_ANOMALY_ML_ROOT = (
    f"{GOLD_LAKEHOUSE_ROOT}/"
    f"Files/ml/pricing_anomaly"
)


TRAINING_FEATURES_PATH = (
    f"{PRICING_ANOMALY_ML_ROOT}/"
    f"training_features"
)


SCORING_FEATURES_PATH = (
    f"{PRICING_ANOMALY_ML_ROOT}/"
    f"scoring_features"
)


FEATURE_PROFILE_PATH = (
    f"{PRICING_ANOMALY_ML_ROOT}/"
    f"feature_profile"
)


print(
    "Fact PO:",
    FACT_PURCHASE_ORDER_PATH
)

print(
    "Training features:",
    TRAINING_FEATURES_PATH
)

print(
    "Scoring features:",
    SCORING_FEATURES_PATH
)

Fact PO: abfss://Project_Procurement@onelake.dfs.fabric.microsoft.com/lh_procurement_gold.Lakehouse/Tables/fact_purchase_order
Training features: abfss://Project_Procurement@onelake.dfs.fabric.microsoft.com/lh_procurement_gold.Lakehouse/Files/ml/pricing_anomaly/training_features
Scoring features: abfss://Project_Procurement@onelake.dfs.fabric.microsoft.com/lh_procurement_gold.Lakehouse/Files/ml/pricing_anomaly/scoring_features


**Read Gold tables**

In [0]:
# ============================================================
# Read Gold analytical tables
# ============================================================

fact_po_df = (
    spark.read
    .format("delta")
    .load(
        FACT_PURCHASE_ORDER_PATH
    )
)


dim_supplier_df = (
    spark.read
    .format("delta")
    .load(
        DIM_SUPPLIER_PATH
    )
)


dim_material_df = (
    spark.read
    .format("delta")
    .load(
        DIM_MATERIAL_PATH
    )
)


dim_category_df = (
    spark.read
    .format("delta")
    .load(
        DIM_CATEGORY_PATH
    )
)


dim_contract_df = (
    spark.read
    .format("delta")
    .load(
        DIM_CONTRACT_PATH
    )
)


dim_date_df = (
    spark.read
    .format("delta")
    .load(
        DIM_DATE_PATH
    )
)


print(
    "fact_purchase_order:",
    f"{fact_po_df.count():,}"
)

print(
    "dim_supplier:",
    f"{dim_supplier_df.count():,}"
)

print(
    "dim_material:",
    f"{dim_material_df.count():,}"
)

print(
    "dim_category:",
    f"{dim_category_df.count():,}"
)

print(
    "dim_contract:",
    f"{dim_contract_df.count():,}"
)

print(
    "dim_date:",
    f"{dim_date_df.count():,}"
)

fact_purchase_order: 75,994
dim_supplier: 500
dim_material: 2,000
dim_category: 20
dim_contract: 800
dim_date: 3,652


**Column-resolution helper**

In [0]:
# ============================================================
# Schema-resolution helper
# ============================================================

def resolve_column_name(
    dataframe,
    logical_name,
    candidates,
    required=True
):

    normalized_columns = {
        column_name.lower():
            column_name

        for column_name
        in dataframe.columns
    }


    for candidate in candidates:

        normalized_candidate = (
            candidate.lower()
        )


        if (
            normalized_candidate
            in normalized_columns
        ):

            return (
                normalized_columns[
                    normalized_candidate
                ]
            )


    if required:

        raise RuntimeError(
            f"Required column '{logical_name}' "
            f"was not found. "
            f"Candidates: {candidates}. "
            f"Available columns: "
            f"{dataframe.columns}"
        )


    return None


print(
    "Schema-resolution helper loaded."
)

Schema-resolution helper loaded.


**Resolve actual Gold schema**

In [0]:
# ============================================================
# Resolve Gold pricing columns
# ============================================================

# ------------------------------------------------------------
# fact_purchase_order core columns
# ------------------------------------------------------------

PO_ITEM_ID_COL = resolve_column_name(
    fact_po_df,
    "POItemID",
    [
        "POItemID",
        "PurchaseOrderItemID",
        "PurchaseOrderLineID"
    ]
)


PO_ID_COL = resolve_column_name(
    fact_po_df,
    "POID",
    [
        "POID",
        "PurchaseOrderID"
    ]
)


SUPPLIER_KEY_COL = resolve_column_name(
    fact_po_df,
    "SupplierKey",
    [
        "SupplierKey"
    ]
)


MATERIAL_KEY_COL = resolve_column_name(
    fact_po_df,
    "MaterialKey",
    [
        "MaterialKey"
    ]
)


CATEGORY_KEY_COL = resolve_column_name(
    fact_po_df,
    "CategoryKey",
    [
        "CategoryKey"
    ]
)


ORDER_DATE_COL = resolve_column_name(
    fact_po_df,
    "OrderDate",
    [
        "OrderDate",
        "PODate"
    ]
)


QUANTITY_COL = resolve_column_name(
    fact_po_df,
    "Quantity",
    [
        "Quantity",
        "OrderedQuantity",
        "OrderQuantity"
    ]
)


UNIT_PRICE_EUR_COL = resolve_column_name(
    fact_po_df,
    "UnitPriceEUR",
    [
        "UnitPriceEUR",
        "POUnitPriceEUR",
        "OrderedUnitPriceEUR"
    ]
)


LINE_AMOUNT_EUR_COL = resolve_column_name(
    fact_po_df,
    "LineAmountEUR",
    [
        "LineAmountEUR",
        "POItemAmountEUR",
        "POValueEUR"
    ]
)


# ------------------------------------------------------------
# Contract-pricing fields already governed in Gold
# ------------------------------------------------------------

CONTRACT_ID_COL = resolve_column_name(
    fact_po_df,
    "ContractID",
    [
        "ContractID"
    ],
    required=False
)


CONTRACT_PRICE_VARIANCE_PCT_COL = resolve_column_name(
    fact_po_df,
    "ContractPriceVariancePercentage",
    [
        "ContractPriceVariancePercentage",
        "ContractPriceVariancePct"
    ]
)


CONTRACT_PRICE_WITHIN_TOLERANCE_COL = resolve_column_name(
    fact_po_df,
    "ContractPriceWithinToleranceFlag",
    [
        "ContractPriceWithinToleranceFlag"
    ],
    required=False
)


VALID_CONTRACT_AT_PO_COL = resolve_column_name(
    fact_po_df,
    "ValidContractAtPOFlag",
    [
        "ValidContractAtPOFlag"
    ],
    required=False
)


HAS_CONTRACT_REFERENCE_COL = resolve_column_name(
    fact_po_df,
    "HasContractReferenceFlag",
    [
        "HasContractReferenceFlag"
    ],
    required=False
)


PRICE_COMPLIANCE_EXCEPTION_COL = resolve_column_name(
    fact_po_df,
    "PriceComplianceExceptionFlag",
    [
        "PriceComplianceExceptionFlag"
    ],
    required=False
)


CONTRACT_VALIDITY_STATUS_COL = resolve_column_name(
    fact_po_df,
    "ContractValidityStatusAtPO",
    [
        "ContractValidityStatusAtPO"
    ],
    required=False
)


# ------------------------------------------------------------
# Dimension descriptive fields
# ------------------------------------------------------------

DIM_SUPPLIER_KEY_COL = resolve_column_name(
    dim_supplier_df,
    "SupplierKey",
    [
        "SupplierKey"
    ]
)


SUPPLIER_ID_COL = resolve_column_name(
    dim_supplier_df,
    "SupplierID",
    [
        "SupplierID"
    ]
)


SUPPLIER_NAME_COL = resolve_column_name(
    dim_supplier_df,
    "SupplierName",
    [
        "SupplierName"
    ]
)


DIM_MATERIAL_KEY_COL = resolve_column_name(
    dim_material_df,
    "MaterialKey",
    [
        "MaterialKey"
    ]
)


MATERIAL_ID_COL = resolve_column_name(
    dim_material_df,
    "MaterialID",
    [
        "MaterialID"
    ]
)


MATERIAL_NAME_COL = resolve_column_name(
    dim_material_df,
    "MaterialName",
    [
        "MaterialName",
        "MaterialDescription",
        "MaterialDesc"
    ],
    required=False
)


DIM_CATEGORY_KEY_COL = resolve_column_name(
    dim_category_df,
    "CategoryKey",
    [
        "CategoryKey"
    ]
)


CATEGORY_ID_COL = resolve_column_name(
    dim_category_df,
    "CategoryID",
    [
        "CategoryID"
    ]
)


CATEGORY_NAME_COL = resolve_column_name(
    dim_category_df,
    "CategoryName",
    [
        "CategoryName",
        "CategoryDescription"
    ]
)


print(
    "Resolved pricing schema:"
)

print(
    "POItemID:",
    PO_ITEM_ID_COL
)

print(
    "POID:",
    PO_ID_COL
)

print(
    "Quantity:",
    QUANTITY_COL
)

print(
    "UnitPriceEUR:",
    UNIT_PRICE_EUR_COL
)

print(
    "LineAmountEUR:",
    LINE_AMOUNT_EUR_COL
)

print(
    "OrderDate:",
    ORDER_DATE_COL
)

print(
    "ContractID:",
    CONTRACT_ID_COL
)

print(
    "Contract Price Variance %:",
    CONTRACT_PRICE_VARIANCE_PCT_COL
)

print(
    "Contract Price Within Tolerance:",
    CONTRACT_PRICE_WITHIN_TOLERANCE_COL
)

print(
    "Valid Contract At PO:",
    VALID_CONTRACT_AT_PO_COL
)

print(
    "Has Contract Reference:",
    HAS_CONTRACT_REFERENCE_COL
)

print(
    "Price Compliance Exception:",
    PRICE_COMPLIANCE_EXCEPTION_COL
)

Resolved pricing schema:
POItemID: POItemID
POID: POID
Quantity: Quantity
UnitPriceEUR: UnitPriceEUR
LineAmountEUR: LineAmountEUR
OrderDate: OrderDate
ContractID: ContractID
Contract Price Variance %: ContractPriceVariancePercentage
Contract Price Within Tolerance: ContractPriceWithinToleranceFlag
Valid Contract At PO: ValidContractAtPOFlag
Has Contract Reference: HasContractReferenceFlag
Price Compliance Exception: PriceComplianceExceptionFlag


**Build canonical PO-item pricing dataset**

In [0]:
# ============================================================
# Build canonical PO-item pricing dataset
# ============================================================

po_work_df = (
    fact_po_df.alias("f")

    .join(
        dim_supplier_df.alias("s"),

        F.col(
            f"f.{SUPPLIER_KEY_COL}"
        )
        ==
        F.col(
            f"s.{DIM_SUPPLIER_KEY_COL}"
        ),

        "left"
    )

    .join(
        dim_material_df.alias("m"),

        F.col(
            f"f.{MATERIAL_KEY_COL}"
        )
        ==
        F.col(
            f"m.{DIM_MATERIAL_KEY_COL}"
        ),

        "left"
    )

    .join(
        dim_category_df.alias("c"),

        F.col(
            f"f.{CATEGORY_KEY_COL}"
        )
        ==
        F.col(
            f"c.{DIM_CATEGORY_KEY_COL}"
        ),

        "left"
    )
)


# ------------------------------------------------------------
# Optional descriptive expressions
# ------------------------------------------------------------

material_name_expression = (
    F.col(
        f"m.{MATERIAL_NAME_COL}"
    )
    if MATERIAL_NAME_COL is not None
    else
    F.lit(None).cast("string")
)


contract_id_expression = (
    F.col(
        f"f.{CONTRACT_ID_COL}"
    )
    if CONTRACT_ID_COL is not None
    else
    F.lit(None).cast("string")
)


contract_within_tolerance_expression = (
    F.col(
        f"f.{CONTRACT_PRICE_WITHIN_TOLERANCE_COL}"
    )
    if CONTRACT_PRICE_WITHIN_TOLERANCE_COL is not None
    else
    F.lit(None).cast("int")
)


valid_contract_expression = (
    F.col(
        f"f.{VALID_CONTRACT_AT_PO_COL}"
    )
    if VALID_CONTRACT_AT_PO_COL is not None
    else
    F.lit(None).cast("int")
)


has_contract_reference_expression = (
    F.col(
        f"f.{HAS_CONTRACT_REFERENCE_COL}"
    )
    if HAS_CONTRACT_REFERENCE_COL is not None
    else
    F.lit(None).cast("int")
)


price_exception_expression = (
    F.col(
        f"f.{PRICE_COMPLIANCE_EXCEPTION_COL}"
    )
    if PRICE_COMPLIANCE_EXCEPTION_COL is not None
    else
    F.lit(None).cast("int")
)


contract_validity_status_expression = (
    F.col(
        f"f.{CONTRACT_VALIDITY_STATUS_COL}"
    )
    if CONTRACT_VALIDITY_STATUS_COL is not None
    else
    F.lit(None).cast("string")
)


# ------------------------------------------------------------
# Canonical ML pricing dataset
# ------------------------------------------------------------

pricing_base_df = (
    po_work_df

    .select(
        F.col(
            f"f.{PO_ITEM_ID_COL}"
        )
        .cast("string")
        .alias(
            "POItemID"
        ),

        F.col(
            f"f.{PO_ID_COL}"
        )
        .cast("string")
        .alias(
            "POID"
        ),

        F.to_date(
            F.col(
                f"f.{ORDER_DATE_COL}"
            )
        )
        .alias(
            "OrderDate"
        ),

        F.col(
            f"s.{SUPPLIER_ID_COL}"
        )
        .cast("string")
        .alias(
            "SupplierID"
        ),

        F.col(
            f"s.{SUPPLIER_NAME_COL}"
        )
        .cast("string")
        .alias(
            "SupplierName"
        ),

        F.col(
            f"m.{MATERIAL_ID_COL}"
        )
        .cast("string")
        .alias(
            "MaterialID"
        ),

        material_name_expression
        .cast("string")
        .alias(
            "MaterialName"
        ),

        F.col(
            f"c.{CATEGORY_ID_COL}"
        )
        .cast("string")
        .alias(
            "CategoryID"
        ),

        F.col(
            f"c.{CATEGORY_NAME_COL}"
        )
        .cast("string")
        .alias(
            "CategoryName"
        ),

        contract_id_expression
        .alias(
            "ContractID"
        ),

        F.col(
            f"f.{QUANTITY_COL}"
        )
        .cast("double")
        .alias(
            "Quantity"
        ),

        F.col(
            f"f.{UNIT_PRICE_EUR_COL}"
        )
        .cast("double")
        .alias(
            "UnitPriceEUR"
        ),

        F.col(
            f"f.{LINE_AMOUNT_EUR_COL}"
        )
        .cast("double")
        .alias(
            "LineAmountEUR"
        ),

        F.col(
            f"f.{CONTRACT_PRICE_VARIANCE_PCT_COL}"
        )
        .cast("double")
        .alias(
            "ContractPriceVariancePct"
        ),

        contract_within_tolerance_expression
        .cast("int")
        .alias(
            "ContractPriceWithinToleranceFlag"
        ),

        valid_contract_expression
        .cast("int")
        .alias(
            "ValidContractAtPOFlag"
        ),

        has_contract_reference_expression
        .cast("int")
        .alias(
            "HasContractReferenceFlag"
        ),

        price_exception_expression
        .cast("int")
        .alias(
            "PriceComplianceExceptionFlag"
        ),

        contract_validity_status_expression
        .cast("string")
        .alias(
            "ContractValidityStatusAtPO"
        )
    )

    .withColumn(
        "OrderYear",
        F.year(
            "OrderDate"
        )
    )

    .filter(
        F.col(
            "OrderDate"
        ).isNotNull()
    )

    .filter(
        F.col(
            "OrderDate"
        )
        <=
        F.lit(
            AS_OF_DATE
        )
    )

    .filter(
        F.col(
            "UnitPriceEUR"
        )
        > 0
    )

    .filter(
        F.col(
            "Quantity"
        )
        > 0
    )

    .filter(
        F.col(
            "MaterialID"
        ).isNotNull()
    )
)


pricing_base_count = (
    pricing_base_df.count()
)


print(
    "Eligible PO-item pricing rows:",
    f"{pricing_base_count:,}"
)

Eligible PO-item pricing rows: 75,994


In [0]:
# ============================================================
# Validate canonical contract-pricing fields
# ============================================================

required_contract_columns = [
    "ContractPriceVariancePct",
    "ContractPriceWithinToleranceFlag",
    "ValidContractAtPOFlag",
    "HasContractReferenceFlag",
    "PriceComplianceExceptionFlag"
]


print(
    "pricing_base_df columns:"
)

print(
    pricing_base_df.columns
)


missing_contract_columns = [
    column_name
    for column_name in required_contract_columns
    if column_name
    not in pricing_base_df.columns
]


if missing_contract_columns:

    raise ValueError(
        "pricing_base_df is missing required "
        "Gold contract-pricing columns: "
        + ", ".join(
            missing_contract_columns
        )
    )


print(
    "\nContract pricing columns successfully "
    "carried into pricing_base_df."
)


display(
    pricing_base_df

    .select(
        "POItemID",
        "ContractID",
        "UnitPriceEUR",
        "ContractPriceVariancePct",
        "ContractPriceWithinToleranceFlag",
        "ValidContractAtPOFlag",
        "HasContractReferenceFlag",
        "PriceComplianceExceptionFlag",
        "ContractValidityStatusAtPO"
    )

    .filter(
        F.col(
            "ContractPriceVariancePct"
        ).isNotNull()
    )

    .limit(20)
)

pricing_base_df columns:
['POItemID', 'POID', 'OrderDate', 'SupplierID', 'SupplierName', 'MaterialID', 'MaterialName', 'CategoryID', 'CategoryName', 'ContractID', 'Quantity', 'UnitPriceEUR', 'LineAmountEUR', 'ContractPriceVariancePct', 'ContractPriceWithinToleranceFlag', 'ValidContractAtPOFlag', 'HasContractReferenceFlag', 'PriceComplianceExceptionFlag', 'ContractValidityStatusAtPO', 'OrderYear']

Contract pricing columns successfully carried into pricing_base_df.


POItemID,ContractID,UnitPriceEUR,ContractPriceVariancePct,ContractPriceWithinToleranceFlag,ValidContractAtPOFlag,HasContractReferenceFlag,PriceComplianceExceptionFlag,ContractValidityStatusAtPO
4500012393-00070,CTR00000182,78242.8,-1.73,1,1,1,0,VALID_CONTRACT
4500012394-00010,CTR00000048,88.04,-0.05,1,1,1,0,VALID_CONTRACT
4500012394-00020,CTR00000048,89.94,2.11,1,1,1,0,VALID_CONTRACT
4500012394-00030,CTR00000033,446.07,2.46,1,1,1,0,VALID_CONTRACT
4500012395-00010,CTR00000563,268.8036,-0.4,1,1,1,0,VALID_CONTRACT
4500012396-00010,CTR00000526,124.4587,-0.91,1,1,1,0,VALID_CONTRACT
4500012396-00040,CTR00000526,128.8011,2.55,1,1,1,0,VALID_CONTRACT
4500012396-00050,CTR00000526,128.7824,2.53,1,1,1,0,VALID_CONTRACT
4500012398-00010,CTR00000334,4220.9699,1.25,1,1,1,0,VALID_CONTRACT
4500012398-00020,CTR00000305,6490.6861,2.61,1,1,1,0,VALID_CONTRACT


**Validate PO-item grain**

In [0]:
# ============================================================
# Validate canonical pricing grain
# ============================================================

duplicate_po_item_count = (
    pricing_base_df

    .groupBy(
        "POItemID"
    )

    .count()

    .filter(
        F.col(
            "count"
        )
        > 1
    )

    .count()
)


null_supplier_count = (
    pricing_base_df

    .filter(
        F.col(
            "SupplierID"
        ).isNull()
    )

    .count()
)


null_category_count = (
    pricing_base_df

    .filter(
        F.col(
            "CategoryID"
        ).isNull()
    )

    .count()
)


if duplicate_po_item_count > 0:

    raise ValueError(
        f"Pricing base contains "
        f"{duplicate_po_item_count:,} "
        f"duplicate POItemID values."
    )


if null_supplier_count > 0:

    raise ValueError(
        f"Pricing base contains "
        f"{null_supplier_count:,} "
        f"rows without SupplierID."
    )


if null_category_count > 0:

    raise ValueError(
        f"Pricing base contains "
        f"{null_category_count:,} "
        f"rows without CategoryID."
    )


print(
    "PO-item grain validation PASSED."
)

print(
    "Unique PO items:",
    f"{pricing_base_count:,}"
)

PO-item grain validation PASSED.
Unique PO items: 75,994


**Define leakage-safe historical windows**

In [0]:
# ============================================================
# Leakage-safe historical windows
#
# Every benchmark contains PREVIOUS transactions only.
# The current PO item is excluded.
# ============================================================

historical_order_columns = [
    F.col(
        "OrderDate"
    ).asc(),

    F.col(
        "POItemID"
    ).asc()
]


material_history_window = (
    Window

    .partitionBy(
        "MaterialID"
    )

    .orderBy(
        *historical_order_columns
    )

    .rowsBetween(
        Window.unboundedPreceding,
        -1
    )
)


supplier_material_history_window = (
    Window

    .partitionBy(
        "SupplierID",
        "MaterialID"
    )

    .orderBy(
        *historical_order_columns
    )

    .rowsBetween(
        Window.unboundedPreceding,
        -1
    )
)


category_history_window = (
    Window

    .partitionBy(
        "CategoryID"
    )

    .orderBy(
        *historical_order_columns
    )

    .rowsBetween(
        Window.unboundedPreceding,
        -1
    )
)


print(
    "Historical benchmark windows configured."
)

print(
    "Current PO item excluded from every benchmark."
)

Historical benchmark windows configured.
Current PO item excluded from every benchmark.


**Material historical pricing features**

In [0]:
# ============================================================
# Historical material-price features
# ============================================================

pricing_features_df = (
    pricing_base_df

    .withColumn(
        "MaterialHistoryCount",

        F.count(
            "UnitPriceEUR"
        ).over(
            material_history_window
        )
    )

    .withColumn(
        "MaterialHistoricalAvgPriceEUR",

        F.avg(
            "UnitPriceEUR"
        ).over(
            material_history_window
        )
    )

    .withColumn(
        "MaterialHistoricalStdDevPriceEUR",

        F.stddev_samp(
            "UnitPriceEUR"
        ).over(
            material_history_window
        )
    )

    .withColumn(
        "MaterialHistoricalMinPriceEUR",

        F.min(
            "UnitPriceEUR"
        ).over(
            material_history_window
        )
    )

    .withColumn(
        "MaterialHistoricalMaxPriceEUR",

        F.max(
            "UnitPriceEUR"
        ).over(
            material_history_window
        )
    )
)


print(
    "Material historical features created."
)

Material historical features created.


**Supplier-material historical features**

In [0]:
# ============================================================
# Supplier-material historical pricing features
# ============================================================

pricing_features_df = (
    pricing_features_df

    .withColumn(
        "SupplierMaterialHistoryCount",

        F.count(
            "UnitPriceEUR"
        ).over(
            supplier_material_history_window
        )
    )

    .withColumn(
        "SupplierMaterialHistoricalAvgPriceEUR",

        F.avg(
            "UnitPriceEUR"
        ).over(
            supplier_material_history_window
        )
    )

    .withColumn(
        "SupplierMaterialHistoricalStdDevPriceEUR",

        F.stddev_samp(
            "UnitPriceEUR"
        ).over(
            supplier_material_history_window
        )
    )

    .withColumn(
        "SupplierMaterialHistoricalMinPriceEUR",

        F.min(
            "UnitPriceEUR"
        ).over(
            supplier_material_history_window
        )
    )

    .withColumn(
        "SupplierMaterialHistoricalMaxPriceEUR",

        F.max(
            "UnitPriceEUR"
        ).over(
            supplier_material_history_window
        )
    )
)


print(
    "Supplier-material historical features created."
)

Supplier-material historical features created.


**Category historical pricing features**

In [0]:
# ============================================================
# Historical category-price features
# ============================================================

pricing_features_df = (
    pricing_features_df

    .withColumn(
        "CategoryHistoryCount",

        F.count(
            "UnitPriceEUR"
        ).over(
            category_history_window
        )
    )

    .withColumn(
        "CategoryHistoricalAvgPriceEUR",

        F.avg(
            "UnitPriceEUR"
        ).over(
            category_history_window
        )
    )

    .withColumn(
        "CategoryHistoricalStdDevPriceEUR",

        F.stddev_samp(
            "UnitPriceEUR"
        ).over(
            category_history_window
        )
    )
)


print(
    "Category historical features created."
)

Category historical features created.


**Price deviation helper expressions**

In [0]:
# ============================================================
# Reusable pricing feature expressions
# ============================================================

def safe_percentage_difference(
    actual_column,
    benchmark_column
):

    return (
        F.when(
            benchmark_column.isNotNull()
            &
            (
                benchmark_column
                > 0
            ),

            (
                (
                    actual_column
                    -
                    benchmark_column
                )
                /
                benchmark_column
            )
            * 100.0
        )
    )


def safe_price_ratio(
    actual_column,
    benchmark_column
):

    return (
        F.when(
            benchmark_column.isNotNull()
            &
            (
                benchmark_column
                > 0
            ),

            actual_column
            /
            benchmark_column
        )
    )


def safe_log_price_ratio(
    actual_column,
    benchmark_column
):

    return (
        F.when(
            benchmark_column.isNotNull()
            &
            (
                benchmark_column
                > 0
            )
            &
            (
                actual_column
                > 0
            ),

            F.log(
                actual_column
                /
                benchmark_column
            )
        )
    )


def safe_z_score(
    actual_column,
    average_column,
    stddev_column
):

    return (
        F.when(
            average_column.isNotNull()
            &
            stddev_column.isNotNull()
            &
            (
                stddev_column
                > 0
            ),

            (
                actual_column
                -
                average_column
            )
            /
            stddev_column
        )
    )


print(
    "Pricing feature expressions loaded."
)

Pricing feature expressions loaded.


**Build material and supplier-material deviations**

In [0]:
# ============================================================
# Material and supplier-material price deviations
# ============================================================

pricing_features_df = (
    pricing_features_df

    # --------------------------------------------------------
    # Material benchmark
    # --------------------------------------------------------

    .withColumn(
        "PriceVsMaterialHistoricalAvgPct",

        safe_percentage_difference(
            F.col(
                "UnitPriceEUR"
            ),

            F.col(
                "MaterialHistoricalAvgPriceEUR"
            )
        )
    )

    .withColumn(
        "PriceToMaterialHistoricalRatio",

        safe_price_ratio(
            F.col(
                "UnitPriceEUR"
            ),

            F.col(
                "MaterialHistoricalAvgPriceEUR"
            )
        )
    )

    .withColumn(
        "LogPriceToMaterialHistoricalRatio",

        safe_log_price_ratio(
            F.col(
                "UnitPriceEUR"
            ),

            F.col(
                "MaterialHistoricalAvgPriceEUR"
            )
        )
    )

    .withColumn(
        "MaterialPriceZScore",

        safe_z_score(
            F.col(
                "UnitPriceEUR"
            ),

            F.col(
                "MaterialHistoricalAvgPriceEUR"
            ),

            F.col(
                "MaterialHistoricalStdDevPriceEUR"
            )
        )
    )


    # --------------------------------------------------------
    # Supplier-material benchmark
    # --------------------------------------------------------

    .withColumn(
        "PriceVsSupplierMaterialHistoricalAvgPct",

        safe_percentage_difference(
            F.col(
                "UnitPriceEUR"
            ),

            F.col(
                "SupplierMaterialHistoricalAvgPriceEUR"
            )
        )
    )

    .withColumn(
        "PriceToSupplierMaterialHistoricalRatio",

        safe_price_ratio(
            F.col(
                "UnitPriceEUR"
            ),

            F.col(
                "SupplierMaterialHistoricalAvgPriceEUR"
            )
        )
    )

    .withColumn(
        "LogPriceToSupplierMaterialHistoricalRatio",

        safe_log_price_ratio(
            F.col(
                "UnitPriceEUR"
            ),

            F.col(
                "SupplierMaterialHistoricalAvgPriceEUR"
            )
        )
    )

    .withColumn(
        "SupplierMaterialPriceZScore",

        safe_z_score(
            F.col(
                "UnitPriceEUR"
            ),

            F.col(
                "SupplierMaterialHistoricalAvgPriceEUR"
            ),

            F.col(
                "SupplierMaterialHistoricalStdDevPriceEUR"
            )
        )
    )
)


print(
    "Material and supplier-material deviations created."
)

Material and supplier-material deviations created.


**16: Category and contract price deviations**

In [0]:
# ============================================================
# Category and governed contract-price features
# ============================================================

pricing_features_df = (
    pricing_features_df

    # --------------------------------------------------------
    # Category benchmark
    # --------------------------------------------------------

    .withColumn(
        "PriceVsCategoryHistoricalAvgPct",

        safe_percentage_difference(
            F.col(
                "UnitPriceEUR"
            ),

            F.col(
                "CategoryHistoricalAvgPriceEUR"
            )
        )
    )

    .withColumn(
        "PriceToCategoryHistoricalRatio",

        safe_price_ratio(
            F.col(
                "UnitPriceEUR"
            ),

            F.col(
                "CategoryHistoricalAvgPriceEUR"
            )
        )
    )

    .withColumn(
        "LogPriceToCategoryHistoricalRatio",

        safe_log_price_ratio(
            F.col(
                "UnitPriceEUR"
            ),

            F.col(
                "CategoryHistoricalAvgPriceEUR"
            )
        )
    )

    .withColumn(
        "CategoryPriceZScore",

        safe_z_score(
            F.col(
                "UnitPriceEUR"
            ),

            F.col(
                "CategoryHistoricalAvgPriceEUR"
            ),

            F.col(
                "CategoryHistoricalStdDevPriceEUR"
            )
        )
    )


    # --------------------------------------------------------
    # Gold-governed contract benchmark
    #
    # ContractPriceVariancePct was calculated upstream using
    # the governed procurement price-compliance logic.
    #
    # Positive:
    # PO price above negotiated contract benchmark
    #
    # Negative:
    # PO price below negotiated contract benchmark
    # --------------------------------------------------------

    .withColumn(
        "AbsoluteContractPriceVariancePct",

        F.abs(
            F.col(
                "ContractPriceVariancePct"
            )
        )
    )
)


print(
    "Category and governed contract-price features created."
)

Category and governed contract-price features created.


**Cell 17**

In [0]:
# ============================================================
# Supplier pricing variance and transaction-scale features
# ============================================================

pricing_features_df = (
    pricing_features_df

    .withColumn(
        "SupplierMaterialHistoricalCVPct",

        F.when(
            F.col(
                "SupplierMaterialHistoricalAvgPriceEUR"
            )
            > 0,

            (
                F.col(
                    "SupplierMaterialHistoricalStdDevPriceEUR"
                )
                /
                F.col(
                    "SupplierMaterialHistoricalAvgPriceEUR"
                )
            )
            * 100.0
        )
    )

    .withColumn(
        "AbsoluteMaterialPriceDeviationPct",

        F.abs(
            F.col(
                "PriceVsMaterialHistoricalAvgPct"
            )
        )
    )

    .withColumn(
        "AbsoluteSupplierMaterialPriceDeviationPct",

        F.abs(
            F.col(
                "PriceVsSupplierMaterialHistoricalAvgPct"
            )
        )
    )

    .withColumn(
        "AbsoluteCategoryPriceDeviationPct",

        F.abs(
            F.col(
                "PriceVsCategoryHistoricalAvgPct"
            )
        )
    )

    .withColumn(
        "LogUnitPriceEUR",

        F.log1p(
            F.col(
                "UnitPriceEUR"
            )
        )
    )

    .withColumn(
        "LogQuantity",

        F.log1p(
            F.col(
                "Quantity"
            )
        )
    )

    .withColumn(
        "LogLineAmountEUR",

        F.log1p(
            F.greatest(
                F.col(
                    "LineAmountEUR"
                ),
                F.lit(
                    0.0
                )
            )
        )
    )
)


print(
    "Supplier variance and scale features created."
)

Supplier variance and scale features created.


Cell 18 Benchmark coverage flag

In [0]:
# ============================================================
# Benchmark availability / reliability flags
# ============================================================

pricing_features_df = (
    pricing_features_df

    .withColumn(
        "HasMaterialHistoryFlag",

        (
            F.col(
                "MaterialHistoryCount"
            )
            >=
            F.lit(
                MIN_MATERIAL_HISTORY_COUNT
            )
        )
        .cast("int")
    )

    .withColumn(
        "HasSupplierMaterialHistoryFlag",

        (
            F.col(
                "SupplierMaterialHistoryCount"
            )
            >=
            F.lit(
                MIN_SUPPLIER_MATERIAL_HISTORY_COUNT
            )
        )
        .cast("int")
    )

    .withColumn(
        "HasCategoryHistoryFlag",

        (
            F.col(
                "CategoryHistoryCount"
            )
            >=
            F.lit(
                MIN_CATEGORY_HISTORY_COUNT
            )
        )
        .cast("int")
    )

    # --------------------------------------------------------
    # Contract benchmark exists only where the governed Gold
    # price comparison produced a variance value.
    # --------------------------------------------------------

    .withColumn(
        "HasContractBenchmarkFlag",

        F.col(
            "ContractPriceVariancePct"
        )
        .isNotNull()
        .cast("int")
    )

    .withColumn(
        "BenchmarkCoverageCount",

        (
            F.col(
                "HasMaterialHistoryFlag"
            )
            +
            F.col(
                "HasSupplierMaterialHistoryFlag"
            )
            +
            F.col(
                "HasCategoryHistoryFlag"
            )
            +
            F.col(
                "HasContractBenchmarkFlag"
            )
        )
    )

    .withColumn(
        "PricingModelEligibleFlag",

        (
            F.col(
                "BenchmarkCoverageCount"
            )
            >= 1
        )
        .cast("int")
    )
)


print(
    "Benchmark coverage flags created."
)

Benchmark coverage flags created.


**Build rule-based anomaly proxy**

In [0]:
# ============================================================
# Transparent rule-based extreme-price proxy
#
# IMPORTANT:
# This is NOT used as an Isolation Forest training target.
#
# It exists only for later model sanity checking because
# synthetic data does not contain a true reviewed anomaly label.
# ============================================================

pricing_features_df = (
    pricing_features_df

    .withColumn(
        "RuleBasedExtremePriceFlag",

        F.when(
            (
                F.col(
                    "AbsoluteContractPriceVariancePct"
                )
                >=
                F.lit(
                    EXTREME_PRICE_VARIANCE_PCT
                )
            )
            |
            (
                F.abs(
                    F.col(
                        "MaterialPriceZScore"
                    )
                )
                >=
                F.lit(
                    EXTREME_Z_SCORE
                )
            )
            |
            (
                F.abs(
                    F.col(
                        "SupplierMaterialPriceZScore"
                    )
                )
                >=
                F.lit(
                    EXTREME_Z_SCORE
                )
            ),

            1
        )

        .otherwise(
            0
        )
    )
)


print(
    "Rule-based pricing diagnostic created."
)

print(
    "It will NOT be used as a model training target."
)

Rule-based pricing diagnostic created.
It will NOT be used as a model training target.


**Inspect benchmark coverage by year**

In [0]:
# ============================================================
# Inspect pricing benchmark coverage by year
# ============================================================

pricing_coverage_by_year_df = (
    pricing_features_df

    .groupBy(
        "OrderYear"
    )

    .agg(
        F.count("*")
        .alias(
            "POItemCount"
        ),

        F.round(
            F.avg(
                F.col(
                    "HasMaterialHistoryFlag"
                )
                .cast("double")
            )
            * 100.0,
            2
        )
        .alias(
            "MaterialHistoryCoveragePct"
        ),

        F.round(
            F.avg(
                F.col(
                    "HasSupplierMaterialHistoryFlag"
                )
                .cast("double")
            )
            * 100.0,
            2
        )
        .alias(
            "SupplierMaterialHistoryCoveragePct"
        ),

        F.round(
            F.avg(
                F.col(
                    "HasCategoryHistoryFlag"
                )
                .cast("double")
            )
            * 100.0,
            2
        )
        .alias(
            "CategoryHistoryCoveragePct"
        ),

        F.round(
            F.avg(
                F.col(
                    "HasContractBenchmarkFlag"
                )
                .cast("double")
            )
            * 100.0,
            2
        )
        .alias(
            "ContractBenchmarkCoveragePct"
        ),

        F.round(
            F.avg(
                F.col(
                    "PricingModelEligibleFlag"
                )
                .cast("double")
            )
            * 100.0,
            2
        )
        .alias(
            "PricingModelEligiblePct"
        ),

        F.round(
            F.avg(
                F.col(
                    "RuleBasedExtremePriceFlag"
                )
                .cast("double")
            )
            * 100.0,
            2
        )
        .alias(
            "RuleBasedExtremePricePct"
        )
    )

    .orderBy(
        "OrderYear"
    )
)


display(
    pricing_coverage_by_year_df
)

OrderYear,POItemCount,MaterialHistoryCoveragePct,SupplierMaterialHistoryCoveragePct,CategoryHistoryCoveragePct,ContractBenchmarkCoveragePct,PricingModelEligiblePct,RuleBasedExtremePricePct
2022,12733,39.43,3.7,96.86,51.54,98.28,11.11
2023,10446,84.7,8.33,100.0,48.05,100.0,8.99
2024,12123,97.4,11.82,100.0,61.56,100.0,8.88
2025,18940,99.75,17.87,100.0,68.23,100.0,8.4
2026,21752,99.99,24.48,100.0,72.11,100.0,9.37


**Cell 21 Inspect engineered pricing features**

In [0]:
# ============================================================
# Inspect engineered pricing features
# ============================================================

display(
    pricing_features_df

    .select(
        "POItemID",
        "POID",
        "OrderDate",

        "SupplierID",
        "SupplierName",

        "MaterialID",
        "MaterialName",

        "CategoryID",
        "CategoryName",

        "ContractID",

        "UnitPriceEUR",

        "MaterialHistoricalAvgPriceEUR",
        "SupplierMaterialHistoricalAvgPriceEUR",
        "CategoryHistoricalAvgPriceEUR",

        "PriceVsMaterialHistoricalAvgPct",
        "PriceVsSupplierMaterialHistoricalAvgPct",
        "PriceVsCategoryHistoricalAvgPct",

        "ContractPriceVariancePct",
        "AbsoluteContractPriceVariancePct",

        "ContractPriceWithinToleranceFlag",
        "ValidContractAtPOFlag",
        "HasContractReferenceFlag",
        "PriceComplianceExceptionFlag",
        "ContractValidityStatusAtPO",

        "MaterialPriceZScore",
        "SupplierMaterialPriceZScore",
        "CategoryPriceZScore",

        "BenchmarkCoverageCount",
        "RuleBasedExtremePriceFlag"
    )

    .orderBy(
        F.desc(
            "OrderDate"
        )
    )

    .limit(50)
)

POItemID,POID,OrderDate,SupplierID,SupplierName,MaterialID,MaterialName,CategoryID,CategoryName,ContractID,UnitPriceEUR,MaterialHistoricalAvgPriceEUR,SupplierMaterialHistoricalAvgPriceEUR,CategoryHistoricalAvgPriceEUR,PriceVsMaterialHistoricalAvgPct,PriceVsSupplierMaterialHistoricalAvgPct,PriceVsCategoryHistoricalAvgPct,ContractPriceVariancePct,AbsoluteContractPriceVariancePct,ContractPriceWithinToleranceFlag,ValidContractAtPOFlag,HasContractReferenceFlag,PriceComplianceExceptionFlag,ContractValidityStatusAtPO,MaterialPriceZScore,SupplierMaterialPriceZScore,CategoryPriceZScore,BenchmarkCoverageCount,RuleBasedExtremePriceFlag
4500001877-00010,4500001877,2026-07-31,SUP000419,Ironwood Advanced Chemicals S.A.,MAT000599,Packaging Material 00060,CAT006,Packaging Materials,null,100.79,26.645070370370373,null,11.271021928025368,278.2688452272945,null,794.2401198722356,null,null,null,0,0,0,NO_CONTRACT_REFERENCE,1.846587382304632,null,3.678554349399268,2,0
4500005505-00020,4500005505,2026-07-31,SUP000472,Cobalt Integrated Systems Group,MAT000936,Packaging Material 00089,CAT006,Packaging Materials,CTR00000690,6.25,8.065456521739131,null,11.287710775540651,-22.50903612022286,null,-44.630048339446034,-1.57,1.57,1,1,1,0,VALID_CONTRACT,-0.4746739601938111,null,-0.2067704989309966,3,0
4500009389-00020,4500009389,2026-07-31,SUP000440,Summit Dynamic Materials Inc.,MAT000434,Packaging Material 00039,CAT006,Packaging Materials,null,0.4757,5.42070810810811,null,11.286771780055927,-91.22439374131834,null,-95.78533163184379,null,null,null,0,0,0,NO_CONTRACT_REFERENCE,-1.3076103192366912,null,-0.44377501940875796,2,0
4500012327-00010,4500012327,2026-07-31,SUP000102,Pioneer Dynamic Mechanical Ltd.,MAT001390,Packaging Material 00138,CAT006,Packaging Materials,CTR00000139,5.7,7.367966666666666,5.67,11.284757044353345,-22.63808649152411,0.5291005291005335,-49.48938663369664,0.53,0.53,1,1,1,0,VALID_CONTRACT,-0.6265592027498796,null,-0.22926137520572148,3,0
4500012327-00020,4500012327,2026-07-31,SUP000102,Pioneer Dynamic Mechanical Ltd.,MAT001783,Packaging Material 00171,CAT006,Packaging Materials,CTR00000139,5.81,10.351515384615384,null,11.283716471026654,-43.8729520835671,null,-48.509872479351884,2.47,2.47,1,1,1,0,VALID_CONTRACT,-0.4295090883145831,null,-0.2247228585291666,3,0
4500014116-00010,4500014116,2026-07-31,SUP000295,Lumina Precision Engineering Group,MAT001405,Packaging Material 00140,CAT006,Packaging Materials,CTR00000052,6.2563,16.343869230769226,6.4190499999999995,11.28269677719822,-61.720814626797235,-2.5354219082262808,-44.549604376112654,-0.49,0.49,1,1,1,0,VALID_CONTRACT,-0.49828359600505234,-2.170058511996107,-0.2063764516498205,4,0
4500014116-00030,4500014116,2026-07-31,SUP000295,Lumina Precision Engineering Group,MAT000005,Packaging Material 00002,CAT006,Packaging Materials,CTR00000052,6.37,52.086140625,6.04,11.281760588563987,-87.77025918303003,5.463576158940398,-43.537181541885445,1.32,1.32,1,1,1,0,VALID_CONTRACT,-0.5594761948764689,null,-0.20168764545011603,3,0
4500014677-00020,4500014677,2026-07-31,SUP000334,Quantum Precision Supply Group,MAT001335,Packaging Material 00131,CAT006,Packaging Materials,null,3.17,5.896535714285712,null,11.28084592178772,-46.239620115927615,null,-71.89927048043896,null,null,null,0,0,0,NO_CONTRACT_REFERENCE,-2.75576805782285,null,-0.33307885302419465,2,0
4500018719-00010,4500018719,2026-07-31,SUP000042,Ironwood Precision Equipment Group,MAT000263,Packaging Material 00023,CAT006,Packaging Materials,null,178.55,32.84683529411764,null,11.279335803388577,443.5835702320326,null,1482.9832812173163,null,null,null,0,0,0,NO_CONTRACT_REFERENCE,2.410543675385801,null,6.869682179953845,2,0
4500018226-00040,4500018226,2026-07-31,SUP000136,Helios Global Mechanical Ltd.,MAT000395,MRO Item 00011,CAT008,"Maintenance, Repair and Operations",CTR00000160,143.6776,89.15534503816797,137.2176,411.7979042852411,61.15421900783484,4.707850887932749,-65.10968159262937,null,null,null,0,1,0,CONTRACT_EXPIRED,0.8319754108899311,0.513109

**Create DB_05 training population**

Training stops at the end of 2025

In [0]:
# ============================================================
# Build historical Isolation Forest training feature store
#
# Training:
# 2022 through 2025
#
# Only price observations with at least one reliable
# benchmark are included.
# ============================================================

training_features_df = (
    pricing_features_df

    .filter(
        F.col(
            "OrderYear"
        )
        <=
        F.lit(
            TRAINING_END_YEAR
        )
    )

    .filter(
        F.col(
            "PricingModelEligibleFlag"
        )
        == 1
    )

    .withColumn(
        "FeatureEngineeringTimestampUTC",

        F.current_timestamp()
    )

    .withColumn(
        "SourceAsOfDate",

        F.lit(
            AS_OF_DATE
        )
        .cast("date")
    )
)


training_feature_count = (
    training_features_df.count()
)


print(
    "Pricing anomaly training rows:",
    f"{training_feature_count:,}"
)

print(
    "Maximum training year:",
    training_features_df
    .agg(
        F.max(
            "OrderYear"
        )
    )
    .first()[0]
)

Pricing anomaly training rows: 54,023
Maximum training year: 2025


**Create 2026 scoring population**

In [0]:
# ============================================================
# Build current 2026 pricing scoring feature store
# ============================================================

scoring_features_df = (
    pricing_features_df

    .filter(
        F.col(
            "OrderYear"
        )
        ==
        F.lit(
            SCORING_YEAR
        )
    )

    .withColumn(
        "FeatureEngineeringTimestampUTC",

        F.current_timestamp()
    )

    .withColumn(
        "SourceAsOfDate",

        F.lit(
            AS_OF_DATE
        )
        .cast("date")
    )
)


scoring_feature_count = (
    scoring_features_df.count()
)


scoring_eligible_count = (
    scoring_features_df

    .filter(
        F.col(
            "PricingModelEligibleFlag"
        )
        == 1
    )

    .count()
)


print(
    "2026 pricing rows:",
    f"{scoring_feature_count:,}"
)

print(
    "2026 model-eligible rows:",
    f"{scoring_eligible_count:,}"
)

print(
    "2026 model eligibility:",
    (
        f"{(
            scoring_eligible_count
            / scoring_feature_count
            * 100.0
        ):.2f}%"
        if scoring_feature_count > 0
        else "N/A"
    )
)

2026 pricing rows: 21,752
2026 model-eligible rows: 21,752
2026 model eligibility: 100.00%


**24: DB_05 Isolation Forest feature contract**

In [0]:
# ============================================================
# DB_05 Isolation Forest feature contract
# ============================================================

PRICING_MODEL_FEATURES = [

    # --------------------------------------------------------
    # Relative historical material pricing
    # --------------------------------------------------------

    "LogPriceToMaterialHistoricalRatio",
    "MaterialPriceZScore",
    "AbsoluteMaterialPriceDeviationPct",


    # --------------------------------------------------------
    # Supplier-material pricing behavior
    # --------------------------------------------------------

    "LogPriceToSupplierMaterialHistoricalRatio",
    "SupplierMaterialPriceZScore",
    "AbsoluteSupplierMaterialPriceDeviationPct",
    "SupplierMaterialHistoricalCVPct",


    # --------------------------------------------------------
    # Category benchmark
    # --------------------------------------------------------

    "LogPriceToCategoryHistoricalRatio",
    "CategoryPriceZScore",
    "AbsoluteCategoryPriceDeviationPct",


    # --------------------------------------------------------
    # Governed contract-price benchmark
    # --------------------------------------------------------

    "ContractPriceVariancePct",
    "AbsoluteContractPriceVariancePct",


    # --------------------------------------------------------
    # Transaction scale
    # --------------------------------------------------------

    "LogUnitPriceEUR",
    "LogQuantity",


    # --------------------------------------------------------
    # Benchmark availability
    # --------------------------------------------------------

    "HasMaterialHistoryFlag",
    "HasSupplierMaterialHistoryFlag",
    "HasCategoryHistoryFlag",
    "HasContractBenchmarkFlag",
    "BenchmarkCoverageCount"
]


print(
    "DB_05 pricing model feature count:",
    len(
        PRICING_MODEL_FEATURES
    )
)


for feature_name in PRICING_MODEL_FEATURES:

    print(
        feature_name
    )

DB_05 pricing model feature count: 19
LogPriceToMaterialHistoricalRatio
MaterialPriceZScore
AbsoluteMaterialPriceDeviationPct
LogPriceToSupplierMaterialHistoricalRatio
SupplierMaterialPriceZScore
AbsoluteSupplierMaterialPriceDeviationPct
SupplierMaterialHistoricalCVPct
LogPriceToCategoryHistoricalRatio
CategoryPriceZScore
AbsoluteCategoryPriceDeviationPct
ContractPriceVariancePct
AbsoluteContractPriceVariancePct
LogUnitPriceEUR
LogQuantity
HasMaterialHistoryFlag
HasSupplierMaterialHistoryFlag
HasCategoryHistoryFlag
HasContractBenchmarkFlag
BenchmarkCoverageCount


**Feature null profile**

Isolation Forest itself does not accept arbitrary nulls, so DB_05 will use a preprocessing pipeline with imputation and robust scaling.

In [0]:
# ============================================================
# Pricing-model feature null profile
# ============================================================

training_rows_for_profile = (
    training_features_df.count()
)


null_profile_rows = []


for feature_name in PRICING_MODEL_FEATURES:

    null_count = (
        training_features_df

        .filter(
            F.col(
                feature_name
            ).isNull()
        )

        .count()
    )


    null_pct = (
        (
            null_count
            / training_rows_for_profile
            * 100.0
        )
        if training_rows_for_profile > 0
        else 0.0
    )


    null_profile_rows.append(
        (
            feature_name,
            int(
                null_count
            ),
            float(
                null_pct
            )
        )
    )


feature_null_profile_df = (
    spark.createDataFrame(
        null_profile_rows,
        [
            "FeatureName",
            "NullCount",
            "NullPct"
        ]
    )

    .orderBy(
        F.desc(
            "NullPct"
        ),
        "FeatureName"
    )
)


display(
    feature_null_profile_df
)

FeatureName,NullCount,NullPct
SupplierMaterialPriceZScore,42590,78.83679173685282
SupplierMaterialHistoricalCVPct,42585,78.82753641967311
AbsoluteSupplierMaterialPriceDeviationPct,30903,57.20341336097589
LogPriceToSupplierMaterialHistoricalRatio,30903,57.20341336097589
AbsoluteContractPriceVariancePct,22056,40.827055143179756
ContractPriceVariancePct,22056,40.827055143179756
MaterialPriceZScore,3660,6.7748921755548555
AbsoluteMaterialPriceDeviationPct,1744,3.228254632286248
LogPriceToMaterialHistoricalRatio,1744,3.228254632286248
CategoryPriceZScore,13,0.024063824667271347


**DB_04 quality gates**

In [0]:
# ============================================================
# DB_04 Pricing Anomaly Feature Engineering quality gates
# ============================================================

training_duplicate_count = (
    training_features_df

    .groupBy(
        "POItemID"
    )

    .count()

    .filter(
        F.col(
            "count"
        )
        > 1
    )

    .count()
)


scoring_duplicate_count = (
    scoring_features_df

    .groupBy(
        "POItemID"
    )

    .count()

    .filter(
        F.col(
            "count"
        )
        > 1
    )

    .count()
)


training_future_year_count = (
    training_features_df

    .filter(
        F.col(
            "OrderYear"
        )
        >
        TRAINING_END_YEAR
    )

    .count()
)


scoring_wrong_year_count = (
    scoring_features_df

    .filter(
        F.col(
            "OrderYear"
        )
        !=
        SCORING_YEAR
    )

    .count()
)


training_invalid_price_count = (
    training_features_df

    .filter(
        F.col(
            "UnitPriceEUR"
        )
        <= 0
    )

    .count()
)


scoring_invalid_price_count = (
    scoring_features_df

    .filter(
        F.col(
            "UnitPriceEUR"
        )
        <= 0
    )

    .count()
)


quality_checks = [

    (
        "Pricing base contains rows",
        pricing_base_count > 0
    ),

    (
        "Pricing base grain is unique by POItemID",
        duplicate_po_item_count == 0
    ),

    (
        "Training feature store contains rows",
        training_feature_count > 0
    ),

    (
        "Training grain is unique by POItemID",
        training_duplicate_count == 0
    ),

    (
        "Training contains no post-2025 rows",
        training_future_year_count == 0
    ),

    (
        "Training prices are positive",
        training_invalid_price_count == 0
    ),

    (
        "2026 scoring population contains rows",
        scoring_feature_count > 0
    ),

    (
        "2026 scoring contains model-eligible rows",
        scoring_eligible_count > 0
    ),

    (
        "Scoring grain is unique by POItemID",
        scoring_duplicate_count == 0
    ),

    (
        "Scoring population contains only 2026",
        scoring_wrong_year_count == 0
    ),

    (
        "Scoring prices are positive",
        scoring_invalid_price_count == 0
    )
]


failed_checks = []


for (
    check_name,
    passed
) in quality_checks:

    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{check_name}"
    )


    if not passed:

        failed_checks.append(
            check_name
        )


if failed_checks:

    raise ValueError(
        "DB_04 quality gate FAILED: "
        +
        "; ".join(
            failed_checks
        )
    )


print(
    "\nDB_04 PRICING FEATURE QUALITY GATE PASSED."
)

PASS | Pricing base contains rows
PASS | Pricing base grain is unique by POItemID
PASS | Training feature store contains rows
PASS | Training grain is unique by POItemID
PASS | Training contains no post-2025 rows
PASS | Training prices are positive
PASS | 2026 scoring population contains rows
PASS | 2026 scoring contains model-eligible rows
PASS | Scoring grain is unique by POItemID
PASS | Scoring population contains only 2026
PASS | Scoring prices are positive

DB_04 PRICING FEATURE QUALITY GATE PASSED.


**Build feature profile metadata**

In [0]:
# ============================================================
# Build pricing feature-store profile
# ============================================================

feature_profile_df = (
    pricing_features_df

    .groupBy(
        "OrderYear"
    )

    .agg(
        F.count("*")
        .cast("long")
        .alias(
            "POItemCount"
        ),

        F.sum(
            "PricingModelEligibleFlag"
        )
        .cast("long")
        .alias(
            "ModelEligiblePOItemCount"
        ),

        F.sum(
            "RuleBasedExtremePriceFlag"
        )
        .cast("long")
        .alias(
            "RuleBasedExtremePriceCount"
        ),

        F.round(
            F.avg(
                F.col(
                    "BenchmarkCoverageCount"
                )
            ),
            4
        )
        .alias(
            "AverageBenchmarkCoverageCount"
        ),

        F.round(
            F.avg(
                F.col(
                    "HasContractBenchmarkFlag"
                )
                .cast("double")
            )
            * 100.0,
            2
        )
        .alias(
            "ContractBenchmarkCoveragePct"
        )
    )

    .withColumn(
        "FeatureEngineeringTimestampUTC",

        F.current_timestamp()
    )

    .withColumn(
        "SourceAsOfDate",

        F.lit(
            AS_OF_DATE
        )
        .cast("date")
    )

    .orderBy(
        "OrderYear"
    )
)


display(
    feature_profile_df
)

OrderYear,POItemCount,ModelEligiblePOItemCount,RuleBasedExtremePriceCount,AverageBenchmarkCoverageCount,ContractBenchmarkCoveragePct,FeatureEngineeringTimestampUTC,SourceAsOfDate
2022,12733,12514,1414,1.9152,51.54,2026-08-13T07:16:37.663476Z,2026-07-31
2023,10446,10446,939,2.4108,48.05,2026-08-13T07:16:37.663476Z,2026-07-31
2024,12123,12123,1076,2.7078,61.56,2026-08-13T07:16:37.663476Z,2026-07-31
2025,18940,18940,1591,2.8585,68.23,2026-08-13T07:16:37.663476Z,2026-07-31
2026,21752,21752,2038,2.9658,72.11,2026-08-13T07:16:37.663476Z,2026-07-31


**Persist DB_04 feature stores**

In [0]:
# ============================================================
# Persist Pricing Anomaly ML feature stores
# ============================================================

(
    training_features_df

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .save(
        TRAINING_FEATURES_PATH
    )
)


(
    scoring_features_df

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .save(
        SCORING_FEATURES_PATH
    )
)


(
    feature_profile_df

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .save(
        FEATURE_PROFILE_PATH
    )
)


print(
    "DB_04 feature stores written successfully."
)

DB_04 feature stores written successfully.


**Persistance validation**

In [0]:
# ============================================================
# Validate DB_04 persisted feature stores
# ============================================================

persisted_training_df = (
    spark.read
    .format("delta")
    .load(
        TRAINING_FEATURES_PATH
    )
)


persisted_scoring_df = (
    spark.read
    .format("delta")
    .load(
        SCORING_FEATURES_PATH
    )
)


persisted_profile_df = (
    spark.read
    .format("delta")
    .load(
        FEATURE_PROFILE_PATH
    )
)


persisted_training_count = (
    persisted_training_df.count()
)


persisted_scoring_count = (
    persisted_scoring_df.count()
)


persisted_profile_count = (
    persisted_profile_df.count()
)


if (
    persisted_training_count
    !=
    training_feature_count
):

    raise ValueError(
        "Pricing training-feature persistence "
        "validation failed."
    )


if (
    persisted_scoring_count
    !=
    scoring_feature_count
):

    raise ValueError(
        "Pricing scoring-feature persistence "
        "validation failed."
    )


if persisted_profile_count <= 0:

    raise ValueError(
        "Pricing feature-profile persistence "
        "validation failed."
    )


print(
    "DB_04 persistence validation PASSED."
)

print(
    "Training rows:",
    f"{persisted_training_count:,}"
)

print(
    "Scoring rows:",
    f"{persisted_scoring_count:,}"
)

print(
    "Feature-profile rows:",
    f"{persisted_profile_count:,}"
)

print(
    "\nDB_04 PRICING ANOMALY "
    "FEATURE ENGINEERING PASSED."
)

DB_04 persistence validation PASSED.
Training rows: 54,023
Scoring rows: 21,752
Feature-profile rows: 5

DB_04 PRICING ANOMALY FEATURE ENGINEERING PASSED.
